In [0]:
%sql
SHOW TABLES IN silver;


In [0]:
%sql
-- Daily revenue with MA7 + WoW + cumulative
CREATE TABLE IF NOT EXISTS gold.daily_revenue AS
WITH base AS (
  SELECT
    to_date(event_ts) AS event_date,
    SUM(CASE WHEN event_type = 'purchase' THEN price ELSE 0 END) AS revenue,
    COUNT_IF(event_type = 'purchase') AS purchases,
    approx_count_distinct(CASE WHEN event_type = 'purchase' THEN user_id END) AS purchasing_users
  FROM silver.events
  GROUP BY to_date(event_ts)
),
calc AS (
  SELECT
    event_date,
    revenue,
    purchases,
    purchasing_users,
    AVG(revenue) OVER (
      ORDER BY event_date
      ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
    ) AS revenue_ma7,
    revenue - LAG(revenue, 7) OVER (ORDER BY event_date) AS revenue_wow_delta,
    CASE
      WHEN LAG(revenue, 7) OVER (ORDER BY event_date) = 0 THEN NULL
      ELSE ROUND(
        (revenue - LAG(revenue, 7) OVER (ORDER BY event_date)) * 100.0
        / LAG(revenue, 7) OVER (ORDER BY event_date),
        2
      )
    END AS revenue_wow_pct,
    SUM(revenue) OVER (ORDER BY event_date ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS revenue_cum
  FROM base
)
SELECT * FROM calc;


In [0]:
%sql
-- Daily funnel (views → carts → purchases)
CREATE TABLE IF NOT EXISTS gold.daily_funnel AS
WITH daily AS (
  SELECT
    to_date(event_ts) AS event_date,
    category_code,
    COUNT_IF(event_type = 'view') AS views,
    COUNT_IF(event_type = 'cart') AS carts,
    COUNT_IF(event_type = 'purchase') AS purchases
  FROM silver.events
  GROUP BY to_date(event_ts), category_code
)
SELECT
  event_date,
  category_code,
  views,
  carts,
  purchases,
  CASE WHEN views = 0 THEN 0 ELSE ROUND(carts * 100.0 / views, 2) END AS view_to_cart_pct,
  CASE WHEN carts = 0 THEN 0 ELSE ROUND(purchases * 100.0 / carts, 2) END AS cart_to_purchase_pct,
  CASE WHEN views = 0 THEN 0 ELSE ROUND(purchases * 100.0 / views, 2) END AS view_to_purchase_pct
FROM daily;

In [0]:
%sql
-- Top products (last 30 days) with revenue + conversion
CREATE OR REPLACE VIEW gold.top_products_30d AS
WITH bounds AS (
  SELECT date_sub(max(to_date(event_ts)), 30) AS start_date
  FROM silver.events
),
base AS (
  SELECT
    product_id,
    brand,
    category_code,
    COUNT_IF(event_type='view') AS views,
    COUNT_IF(event_type='purchase') AS purchases,
    SUM(CASE WHEN event_type='purchase' THEN price ELSE 0 END) AS revenue
  FROM silver.events
  WHERE to_date(event_ts) >= (SELECT start_date FROM bounds)
  GROUP BY product_id, brand, category_code
)
SELECT
  product_id,
  brand,
  category_code,
  views,
  purchases,
  revenue,
  CASE WHEN views = 0 THEN 0 ELSE ROUND(purchases * 100.0 / views, 2) END AS conversion_rate
FROM base
WHERE purchases > 0
ORDER BY revenue DESC
LIMIT 200;


In [0]:
%sql
-- Customer value table (LTV + tiers)
CREATE TABLE IF NOT EXISTS gold.customer_value AS
WITH user_spend AS (
  SELECT
    user_id,
    COUNT_IF(event_type='purchase') AS purchase_cnt,
    SUM(CASE WHEN event_type='purchase' THEN price ELSE 0 END) AS total_spent,
    MIN(to_date(event_ts)) AS first_seen,
    MAX(to_date(event_ts)) AS last_seen
  FROM silver.events
  GROUP BY user_id
),
tiers AS (
  SELECT
    user_id,
    purchase_cnt,
    total_spent,
    first_seen,
    last_seen,
    CASE
      WHEN purchase_cnt >= 10 THEN 'VIP'
      WHEN purchase_cnt >= 5 THEN 'Loyal'
      WHEN purchase_cnt >= 1 THEN 'Buyer'
      ELSE 'Visitor'
    END AS tier
  FROM user_spend
)
SELECT * FROM tiers;


In [0]:
%sql
-- Cohort retention (first purchase month)
CREATE OR REPLACE VIEW gold.cohort_retention AS
WITH purchases AS (
  SELECT user_id, to_date(event_ts) AS d
  FROM silver.events
  WHERE event_type='purchase'
),
cohort AS (
  SELECT user_id, date_trunc('month', MIN(d)) AS cohort_month
  FROM purchases
  GROUP BY user_id
),
activity AS (
  SELECT
    c.cohort_month,
    date_trunc('month', p.d) AS activity_month,
    COUNT(DISTINCT p.user_id) AS active_users
  FROM purchases p
  JOIN cohort c ON p.user_id = c.user_id
  GROUP BY c.cohort_month, date_trunc('month', p.d)
),
cohort_size AS (
  SELECT cohort_month, COUNT(*) AS cohort_users
  FROM cohort
  GROUP BY cohort_month
)
SELECT
  a.cohort_month,
  a.activity_month,
  a.active_users,
  s.cohort_users,
  ROUND(a.active_users * 100.0 / s.cohort_users, 2) AS retention_pct,
  months_between(a.activity_month, a.cohort_month) AS months_from_cohort
FROM activity a
JOIN cohort_size s USING (cohort_month)
ORDER BY cohort_month, activity_month;


In [0]:
%sql
-- RFM-lite scoring (Recency, Frequency, Monetary)
CREATE OR REPLACE VIEW gold.rfm AS
WITH p AS (
  SELECT
    user_id,
    MAX(to_date(event_ts)) AS last_purchase_date,
    COUNT(*) AS frequency,
    SUM(price) AS monetary
  FROM silver.events
  WHERE event_type='purchase'
  GROUP BY user_id
),
ref AS (
  SELECT MAX(to_date(event_ts)) AS as_of_date FROM silver.events
),
scored AS (
  SELECT
    p.*,
    datediff((SELECT as_of_date FROM ref), last_purchase_date) AS recency_days,
    ntile(5) OVER (ORDER BY datediff((SELECT as_of_date FROM ref), last_purchase_date) ASC) AS r_score,
    ntile(5) OVER (ORDER BY frequency DESC) AS f_score,
    ntile(5) OVER (ORDER BY monetary DESC) AS m_score
  FROM p
)
SELECT
  user_id,
  recency_days,
  frequency,
  monetary,
  r_score, f_score, m_score,
  CONCAT(r_score, f_score, m_score) AS rfm_segment
FROM scored;


In [0]:
%sql
-- Dashboard queries (ready to paste into Databricks SQL)
  -- Revenue trend (filterable)
  SELECT *
  FROM gold.daily_revenue
  WHERE event_date BETWEEN {{start_date}} AND {{end_date}}
  ORDER BY event_date;

  -- Funnel by category (filterable)
  SELECT
  category_code,
  SUM(views) AS views,
  SUM(carts) AS carts,
  SUM(purchases) AS purchases,
  ROUND(SUM(purchases) * 100.0 / NULLIF(SUM(views),0), 2) AS view_to_purchase_pct
  FROM gold.daily_funnel
  WHERE event_date BETWEEN {{start_date}} AND {{end_date}}
  GROUP BY category_code
  ORDER BY view_to_purchase_pct DESC;

  -- Top products (filterable)
  SELECT *
  FROM gold.top_products_30d
  WHERE category_code = COALESCE({{category_code}}, category_code)
  ORDER BY revenue DESC
  LIMIT 100;


In [0]:
%sql
select * from gold.rfm limit 10;

In [0]:
%sql
-- Gold semantic layer: one view to rule metrics
  -- This view standardizes the definitions that can be  reused everywhere: revenue, purchases, AOV, purchasing users, conversion, etc.
CREATE OR REPLACE VIEW gold.v_event_metrics AS
SELECT
  to_date(event_ts) AS event_date,
  category_code,
  brand,
  product_id,

  -- Core counts
  COUNT_IF(event_type = 'view')     AS views,
  COUNT_IF(event_type = 'cart')     AS carts,
  COUNT_IF(event_type = 'purchase') AS purchases,

  -- Revenue only from purchases
  SUM(CASE WHEN event_type = 'purchase' THEN price ELSE 0 END) AS revenue,

  -- Users
  approx_count_distinct(user_id) AS active_users,
  approx_count_distinct(CASE WHEN event_type = 'purchase' THEN user_id END) AS purchasing_users,

  -- Derived metrics (safe divisions)
  CASE WHEN COUNT_IF(event_type='view') = 0 THEN 0
       ELSE ROUND(COUNT_IF(event_type='purchase') * 100.0 / COUNT_IF(event_type='view'), 2)
  END AS conversion_rate,

  CASE WHEN COUNT_IF(event_type='purchase') = 0 THEN 0
       ELSE ROUND(SUM(CASE WHEN event_type='purchase' THEN price ELSE 0 END) / COUNT_IF(event_type='purchase'), 2)
  END AS avg_order_value

FROM silver.events
GROUP BY
  to_date(event_ts), category_code, brand, product_id;


In [0]:
%sql
-- Dashboard layer views built from semantic layer (clean + consistent)
    -- Revenue trend (daily) with MA7 + WoW
CREATE OR REPLACE VIEW gold.v_revenue_trend AS
WITH daily AS (
  SELECT
    event_date,
    SUM(revenue) AS revenue,
    SUM(purchases) AS purchases,
    SUM(purchasing_users) AS purchasing_users
  FROM gold.v_event_metrics
  GROUP BY event_date
)
SELECT
  event_date,
  revenue,
  purchases,
  purchasing_users,
  AVG(revenue) OVER (ORDER BY event_date ROWS BETWEEN 6 PRECEDING AND CURRENT ROW) AS revenue_ma7,
  revenue - LAG(revenue, 7) OVER (ORDER BY event_date) AS revenue_wow_delta,
  CASE
    WHEN LAG(revenue, 7) OVER (ORDER BY event_date) IS NULL OR LAG(revenue, 7) OVER (ORDER BY event_date) = 0 THEN NULL
    ELSE ROUND((revenue - LAG(revenue, 7) OVER (ORDER BY event_date)) * 100.0 / LAG(revenue, 7) OVER (ORDER BY event_date), 2)
  END AS revenue_wow_pct
FROM daily
ORDER BY event_date;


In [0]:
%sql
-- Funnel (Category Level)
CREATE OR REPLACE VIEW gold.v_funnel_by_category AS
SELECT
  event_date,
  category_code,
  SUM(views) AS views,
  SUM(carts) AS carts,
  SUM(purchases) AS purchases,
  CASE WHEN SUM(views) = 0 THEN 0 ELSE ROUND(SUM(carts) * 100.0 / SUM(views), 2) END AS view_to_cart_pct,
  CASE WHEN SUM(carts) = 0 THEN 0 ELSE ROUND(SUM(purchases) * 100.0 / SUM(carts), 2) END AS cart_to_purchase_pct,
  CASE WHEN SUM(views) = 0 THEN 0 ELSE ROUND(SUM(purchases) * 100.0 / SUM(views), 2) END AS view_to_purchase_pct,
  SUM(revenue) AS revenue
FROM gold.v_event_metrics
GROUP BY event_date, category_code;

In [0]:
%sql
-- Top products (with a smart 30-day window based on data max date)
CREATE OR REPLACE VIEW gold.v_top_products_30d AS
WITH bounds AS (
  SELECT date_sub(max(event_date), 30) AS start_date
  FROM gold.v_event_metrics
),
base AS (
  SELECT
    product_id,
    brand,
    category_code,
    SUM(views) AS views,
    SUM(purchases) AS purchases,
    SUM(revenue) AS revenue
  FROM gold.v_event_metrics
  WHERE event_date >= (SELECT start_date FROM bounds)
  GROUP BY product_id, brand, category_code
)
SELECT
  product_id,
  brand,
  category_code,
  views,
  purchases,
  revenue,
  CASE WHEN views = 0 THEN 0 ELSE ROUND(purchases * 100.0 / views, 2) END AS conversion_rate,
  CASE WHEN purchases = 0 THEN 0 ELSE ROUND(revenue / purchases, 2) END AS avg_order_value
FROM base
WHERE purchases > 0
ORDER BY revenue DESC
LIMIT 200;

In [0]:
%sql
-- Customer tiers using percentiles (instead of arbitrary cutoffs)
CREATE OR REPLACE VIEW gold.v_customer_value AS
WITH user_spend AS (
  SELECT
    user_id,
    COUNT_IF(event_type='purchase') AS purchase_cnt,
    SUM(CASE WHEN event_type='purchase' THEN price ELSE 0 END) AS total_spent,
    MAX(to_date(event_ts)) AS last_purchase_date
  FROM silver.events
  GROUP BY user_id
),
scored AS (
  SELECT
    *,
    percent_rank() OVER (ORDER BY total_spent) AS spend_rank,
    percent_rank() OVER (ORDER BY purchase_cnt) AS freq_rank
  FROM user_spend
  WHERE purchase_cnt > 0
)
SELECT
  user_id,
  purchase_cnt,
  total_spent,
  last_purchase_date,
  CASE
    WHEN spend_rank >= 0.90 AND freq_rank >= 0.80 THEN 'VIP'
    WHEN spend_rank >= 0.70 OR  freq_rank >= 0.70 THEN 'Loyal'
    ELSE 'Regular'
  END AS tier
FROM scored;

CREATE OR REPLACE VIEW gold.v_data_health AS
SELECT
  to_date(event_ts) AS event_date,
  COUNT(*) AS total_rows,
  SUM(CASE WHEN user_id IS NULL THEN 1 ELSE 0 END) AS null_user_id,
  SUM(CASE WHEN product_id IS NULL THEN 1 ELSE 0 END) AS null_product_id,
  SUM(CASE WHEN price IS NULL THEN 1 ELSE 0 END) AS null_price,
  SUM(CASE WHEN price < 0 THEN 1 ELSE 0 END) AS negative_price,
  ROUND(SUM(CASE WHEN user_id IS NULL THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS null_user_pct
FROM silver.events
GROUP BY to_date(event_ts)
ORDER BY event_date;